In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/ai4i2020.csv')

df.head()


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [2]:
print("Shape:", df.shape)


Shape: (10000, 14)


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   UDI                      10000 non-null  int64  
 1   Product ID               10000 non-null  str    
 2   Type                     10000 non-null  str    
 3   Air temperature [K]      10000 non-null  float64
 4   Process temperature [K]  10000 non-null  float64
 5   Rotational speed [rpm]   10000 non-null  int64  
 6   Torque [Nm]              10000 non-null  float64
 7   Tool wear [min]          10000 non-null  int64  
 8   Machine failure          10000 non-null  int64  
 9   TWF                      10000 non-null  int64  
 10  HDF                      10000 non-null  int64  
 11  PWF                      10000 non-null  int64  
 12  OSF                      10000 non-null  int64  
 13  RNF                      10000 non-null  int64  
dtypes: float64(3), int64(9), str(2)
me

In [4]:
df.isnull().sum()


UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

In [5]:
df.drop(columns=['UDI', 'Product ID'], inplace=True)


In [7]:
print(df.columns)


Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'],
      dtype='str')


In [8]:
df.columns = df.columns.str.strip()
print(df.columns)


Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'],
      dtype='str')


In [10]:
for col in df.columns:
    print(f"'{col}'")


'Type'
'Air temperature [K]'
'Process temperature [K]'
'Rotational speed [rpm]'
'Torque [Nm]'
'Tool wear [min]'
'Machine failure'
'TWF'
'HDF'
'PWF'
'OSF'
'RNF'


In [12]:
df.columns = df.columns.str.strip()


In [14]:
df.columns

Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
       'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF'],
      dtype='str')

In [16]:
# create 'Failure Type' from one-hot failure columns (TWF, HDF, PWF, OSF, RNF)
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']

# sanity check: ensure required columns exist
missing = [c for c in failure_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing expected failure columns in df: {missing}")

flags = df[failure_cols].fillna(0).astype(int)

# rows with more than one flag set -> mark as 'Multiple'
multi_mask = flags.sum(axis=1) > 1
# rows with no flag set -> mark as 'No Failure'
no_mask = flags.sum(axis=1) == 0

# default: use the flag column name with max value (first occurrence if ties)
df['Failure Type'] = flags.idxmax(axis=1)

# override special cases
df.loc[no_mask, 'Failure Type'] = 'No Failure'
df.loc[multi_mask, 'Failure Type'] = 'Multiple'

# optional: map short codes to readable names
mapping = {
    'TWF': 'Tool wear failure',
    'HDF': 'Heat dissipation failure',
    'PWF': 'Power failure',
    'OSF': 'Overstrain failure',
    'RNF': 'Random failure'
}
df['Failure Type'] = df['Failure Type'].replace(mapping)

# show result
print(df['Failure Type'].value_counts())

Failure Type
No Failure                  9652
Heat dissipation failure     106
Power failure                 80
Overstrain failure            78
Tool wear failure             42
Multiple                      24
Random failure                18
Name: count, dtype: int64


In [17]:
df['Failure Type'].value_counts()


Failure Type
No Failure                  9652
Heat dissipation failure     106
Power failure                 80
Overstrain failure            78
Tool wear failure             42
Multiple                      24
Random failure                18
Name: count, dtype: int64

In [18]:
df = df[df['Failure Type'] != 'Multiple']


In [19]:
df['Failure Type'].value_counts()


Failure Type
No Failure                  9652
Heat dissipation failure     106
Power failure                 80
Overstrain failure            78
Tool wear failure             42
Random failure                18
Name: count, dtype: int64

In [20]:
df.drop(columns=['TWF','HDF','PWF','OSF','RNF'], inplace=True)


In [21]:
df.drop(columns=['Machine failure'], inplace=True)


In [22]:
df['Temp_Diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Torque_Speed'] = df['Torque [Nm]'] * df['Rotational speed [rpm]']


In [23]:
X = df.drop('Failure Type', axis=1)
y = df['Failure Type']


In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [25]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


Failure Type
No Failure                  0.967544
Heat dissipation failure    0.010652
Power failure               0.008020
Overstrain failure          0.007769
Tool wear failure           0.004261
Random failure              0.001754
Name: proportion, dtype: float64
Failure Type
No Failure                  0.967435
Heat dissipation failure    0.010521
Power failure               0.008016
Overstrain failure          0.008016
Tool wear failure           0.004008
Random failure              0.002004
Name: proportion, dtype: float64


In [26]:
print(X.columns)


Index(['Type', 'Air temperature [K]', 'Process temperature [K]',
       'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temp_Diff',
       'Torque_Speed'],
      dtype='str')


In [27]:
categorical_features = ['Type']

numerical_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Temp_Diff',
    'Torque_Speed'
]


In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)


In [29]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE


In [30]:
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(random_state=42))
])


In [31]:
pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The

In [32]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))


                          precision    recall  f1-score   support

Heat dissipation failure       0.91      1.00      0.95        21
              No Failure       0.99      0.98      0.99      1931
      Overstrain failure       0.92      0.75      0.83        16
           Power failure       1.00      1.00      1.00        16
          Random failure       0.00      0.00      0.00         4
       Tool wear failure       0.00      0.00      0.00         8

                accuracy                           0.98      1996
               macro avg       0.64      0.62      0.63      1996
            weighted avg       0.98      0.98      0.98      1996



In [33]:
from sklearn.ensemble import RandomForestClassifier

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        random_state=42,
        class_weight='balanced'
    ))
])


In [34]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(classification_report(y_test, y_pred))


                          precision    recall  f1-score   support

Heat dissipation failure       1.00      1.00      1.00        21
              No Failure       0.99      1.00      0.99      1931
      Overstrain failure       1.00      0.50      0.67        16
           Power failure       1.00      1.00      1.00        16
          Random failure       0.00      0.00      0.00         4
       Tool wear failure       0.00      0.00      0.00         8

                accuracy                           0.99      1996
               macro avg       0.66      0.58      0.61      1996
            weighted avg       0.98      0.99      0.99      1996



/Users/adityachauhan/factoryshield/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/adityachauhan/factoryshield/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/adityachauhan/factoryshield/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

In [1]:
import sys
print(sys.version)


3.11.14 (main, Oct  9 2025, 16:16:55) [Clang 17.0.0 (clang-1700.6.3.2)]


In [2]:
from xgboost import XGBClassifier


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Reload dataset
df = pd.read_csv('../data/ai4i2020.csv')

# Clean column names
df.columns = df.columns.str.strip()

# Create Failure Type from one-hot columns
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
flags = df[failure_cols].fillna(0).astype(int)

df['Failure Type'] = flags.idxmax(axis=1)
df.loc[flags.sum(axis=1) == 0, 'Failure Type'] = 'No Failure'

mapping = {
    'TWF': 'Tool wear failure',
    'HDF': 'Heat dissipation failure',
    'PWF': 'Power failure',
    'OSF': 'Overstrain failure',
    'RNF': 'Random failure'
}
df['Failure Type'] = df['Failure Type'].replace(mapping)

# Remove unwanted classes and leakage
df = df[df['Failure Type'] != 'Multiple']
df.drop(columns=['TWF','HDF','PWF','OSF','RNF','Machine failure'], inplace=True)

# Feature Engineering
df['Temp_Diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Torque_Speed'] = df['Torque [Nm]'] * df['Rotational speed [rpm]']

# Split
X = df.drop('Failure Type', axis=1)
y = df['Failure Type']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Data ready.")


Data ready.


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

class_names = le.classes_
print(class_names)


['Heat dissipation failure' 'No Failure' 'Overstrain failure'
 'Power failure' 'Random failure' 'Tool wear failure']


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = ['Type']

numerical_features = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]',
    'Temp_Diff',
    'Torque_Speed'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

print("Preprocessor ready.")


Preprocessor ready.


In [8]:
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline

pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        objective='multi:softprob',
        num_class=len(class_names),
        eval_metric='mlogloss',
        random_state=42,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8
    ))
])


In [10]:
pipeline_xgb.fit(X_train, y_train_enc)


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The give

In [11]:
from sklearn.metrics import classification_report

y_pred_enc = pipeline_xgb.predict(X_test)
y_pred = le.inverse_transform(y_pred_enc)

print(classification_report(y_test, y_pred))


                          precision    recall  f1-score   support

Heat dissipation failure       0.92      0.96      0.94        23
              No Failure       0.99      1.00      0.99      1930
      Overstrain failure       0.85      0.69      0.76        16
           Power failure       0.94      0.83      0.88        18
          Random failure       0.00      0.00      0.00         4
       Tool wear failure       1.00      0.11      0.20         9

                accuracy                           0.99      2000
               macro avg       0.78      0.60      0.63      2000
            weighted avg       0.99      0.99      0.99      2000



/Users/adityachauhan/factoryshield/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/adityachauhan/factoryshield/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/adityachauhan/factoryshield/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

In [12]:
# Create binary target for Stage 1
df_stage1 = df.copy()

df_stage1['Failure_Binary'] = df_stage1['Failure Type'].apply(
    lambda x: 0 if x == 'No Failure' else 1
)

print(df_stage1['Failure_Binary'].value_counts())


Failure_Binary
0    9652
1     348
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split

X1 = df_stage1.drop(['Failure Type', 'Failure_Binary'], axis=1)
y1 = df_stage1['Failure_Binary']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1,
    test_size=0.2,
    stratify=y1,
    random_state=42
)

print("Train size:", y1_train.value_counts())
print("Test size:", y1_test.value_counts())


Train size: Failure_Binary
0    7722
1     278
Name: count, dtype: int64
Test size: Failure_Binary
0    1930
1      70
Name: count, dtype: int64


In [14]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

pipeline_stage1 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('model', RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced',
        random_state=42
    ))
])


In [15]:
pipeline_stage1.fit(X1_train, y1_train)



,steps,"[('preprocessor', ...), ('smote', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The

In [16]:
from sklearn.metrics import classification_report, confusion_matrix

y1_pred = pipeline_stage1.predict(X1_test)

print(classification_report(y1_test, y1_pred))
print("Confusion Matrix:")
print(confusion_matrix(y1_test, y1_pred))


              precision    recall  f1-score   support

           0       0.99      0.99      0.99      1930
           1       0.79      0.86      0.82        70

    accuracy                           0.99      2000
   macro avg       0.89      0.92      0.91      2000
weighted avg       0.99      0.99      0.99      2000

Confusion Matrix:
[[1914   16]
 [  10   60]]


In [17]:
import numpy as np
from sklearn.metrics import classification_report

# Get probabilities
y1_proba = pipeline_stage1.predict_proba(X1_test)[:, 1]

# Lower threshold
threshold = 0.35
y1_pred_adjusted = (y1_proba > threshold).astype(int)

print(classification_report(y1_test, y1_pred_adjusted))
print("Confusion Matrix:")
print(confusion_matrix(y1_test, y1_pred_adjusted))


              precision    recall  f1-score   support

           0       1.00      0.98      0.99      1930
           1       0.58      0.91      0.71        70

    accuracy                           0.97      2000
   macro avg       0.79      0.95      0.85      2000
weighted avg       0.98      0.97      0.98      2000

Confusion Matrix:
[[1884   46]
 [   6   64]]


In [18]:
# Only failure samples
df_stage2 = df_stage1[df_stage1['Failure_Binary'] == 1].copy()

print(df_stage2['Failure Type'].value_counts())


Failure Type
Heat dissipation failure    115
Power failure                91
Overstrain failure           78
Tool wear failure            46
Random failure               18
Name: count, dtype: int64


In [19]:
X2 = df_stage2.drop(['Failure Type', 'Failure_Binary'], axis=1)
y2 = df_stage2['Failure Type']

from sklearn.model_selection import train_test_split

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2,
    test_size=0.2,
    stratify=y2,
    random_state=42
)

print("Train distribution:")
print(y2_train.value_counts())

print("\nTest distribution:")
print(y2_test.value_counts())


Train distribution:
Failure Type
Heat dissipation failure    92
Power failure               73
Overstrain failure          62
Tool wear failure           37
Random failure              14
Name: count, dtype: int64

Test distribution:
Failure Type
Heat dissipation failure    23
Power failure               18
Overstrain failure          16
Tool wear failure            9
Random failure               4
Name: count, dtype: int64


In [20]:
from sklearn.preprocessing import LabelEncoder

le2 = LabelEncoder()
y2_train_enc = le2.fit_transform(y2_train)
y2_test_enc = le2.transform(y2_test)

print(le2.classes_)


['Heat dissipation failure' 'Overstrain failure' 'Power failure'
 'Random failure' 'Tool wear failure']


In [21]:
from xgboost import XGBClassifier
from imblearn.pipeline import Pipeline

pipeline_stage2 = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(
        objective='multi:softprob',
        num_class=len(le2.classes_),
        eval_metric='mlogloss',
        random_state=42,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05
    ))
])


In [22]:
pipeline_stage2.fit(X2_train, y2_train_enc)


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The give

In [23]:
from sklearn.metrics import classification_report

y2_pred_enc = pipeline_stage2.predict(X2_test)
y2_pred = le2.inverse_transform(y2_pred_enc)

print(classification_report(y2_test, y2_pred))


                          precision    recall  f1-score   support

Heat dissipation failure       0.96      1.00      0.98        23
      Overstrain failure       1.00      0.81      0.90        16
           Power failure       0.95      1.00      0.97        18
          Random failure       0.75      0.75      0.75         4
       Tool wear failure       0.90      1.00      0.95         9

                accuracy                           0.94        70
               macro avg       0.91      0.91      0.91        70
            weighted avg       0.95      0.94      0.94        70



In [24]:
import joblib
import os

# Create models directory if not exists
os.makedirs("../models", exist_ok=True)

# Save Stage 1
joblib.dump(pipeline_stage1, "../models/stage1_binary_model.pkl")

# Save Stage 2
joblib.dump(pipeline_stage2, "../models/stage2_multiclass_model.pkl")

# Save LabelEncoder for Stage 2
joblib.dump(le2, "../models/stage2_label_encoder.pkl")

print("Models saved successfully.")


Models saved successfully.
